# AC-Stark shift in the current three-state transmon model

This standalone notebook compares a **40 us constant square pulse** with **10 us root-Lorentzian and echo-root-Lorentzian pulses** in the repository's three-level Lindblad model over a **100 MHz total detuning domain**, from $-50$ to $+50$ MHz. It starts in $|g\rangle$ and records the final populations of $|g\rangle$, $|e\rangle$, and $|f\rangle$.

Current measurement-linked parameters are used: $T_1=51.24$ us, effective $T_2=7.31$ us ($T_\phi=7.87$ us), and transmon anharmonicity $\alpha/2\pi=-216$ MHz. The model is in the rotating-wave approximation, so this isolates the multilevel dressed-state (AC-Stark) shift and does not include the counter-rotating Bloch-Siegert shift.

**Axis convention:** the entire project uses the conventional drive detuning $\Delta/2\pi=f_d-f_{01}$. Negative transmon anharmonicity therefore places the $g\leftrightarrow f$ two-photon feature at $\alpha/(4\pi)=-108$ MHz, on the left.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'echospec').is_dir())
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.codex_tmp' / 'mpl'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar

from echospec.figures import FigureVariant, apply_figure_style, save_figure
from echospec.simulation.qutrit import simulate_qutrit_map

apply_figure_style(FigureVariant.PAPER)

In [2]:
# Requested scan and current measurement-linked model parameters.
SQUARE_DURATION_US = 40.0
SHAPED_DURATION_US = 10.0
DETUNING_DOMAIN_MHZ = 100.0
DETUNING_POINTS = 101
MODEL_DETUNING_MHZ = np.linspace(
    -DETUNING_DOMAIN_MHZ / 2, DETUNING_DOMAIN_MHZ / 2, DETUNING_POINTS
)
RABI_MHZ = np.linspace(0.0, 60.0, 31)

T1_US = 51.24
T_PHI_US = 7.87
ANHARMONICITY_MHZ = -216.0
SHAPED_STEPS_PER_HALF = max(7000, int(np.ceil(7000 * DETUNING_DOMAIN_MHZ / 300.0)))
SQUARE_STEPS_PER_HALF = int(SHAPED_STEPS_PER_HALF * SQUARE_DURATION_US / SHAPED_DURATION_US)
SHAPED_CUTOFF = 0.002
SHAPED_ORDER = 0.5

print(f'Detuning: {MODEL_DETUNING_MHZ[0]:g} to {MODEL_DETUNING_MHZ[-1]:g} MHz '      f'({DETUNING_POINTS} points, {DETUNING_DOMAIN_MHZ:g} MHz total)')
print(f'Pulse lengths: square {SQUARE_DURATION_US:g} us; shaped {SHAPED_DURATION_US:g} us')
print(f'Peak Rabi range: {RABI_MHZ[0]:g} to {RABI_MHZ[-1]:g} MHz')
print(f'Steps per half: square {SQUARE_STEPS_PER_HALF}; shaped {SHAPED_STEPS_PER_HALF}')
print(f'Shaped pulses: root-Lorentzian order {SHAPED_ORDER:g}, endpoint cutoff {SHAPED_CUTOFF:g}')

Detuning: -50 to 50 MHz (101 points, 100 MHz total)
Pulse lengths: square 40 us; shaped 10 us
Peak Rabi range: 0 to 60 MHz
Steps per half: square 28000; shaped 7000
Shaped pulses: root-Lorentzian order 0.5, endpoint cutoff 0.002


## Main-paper AC-Stark figure

The Letter uses the compact square comparison below. Run this cell to regenerate the exact PDF, PNG, and SVG included by the main manuscript. The plotting and center-extraction implementation is shared with `scripts/make_main_ac_stark_shifts.py`, so the notebook and submission figure stay identical.

In [3]:
from scripts import make_main_ac_stark_shifts as paper_ac_stark

paper_ac_stark.main()
plt.show()

Saved:
  figures/paper/04_main_ac_stark_shifts_square.pdf
  figures/paper/04_main_ac_stark_shifts_square.png
  figures/paper/04_main_ac_stark_shifts_square.svg


/var/folders/5l/mx_yndbx4sq978lx9ht2346r0000gn/T/ipykernel_68016/2416300457.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Time-domain three-state simulation

The dense Runge--Kutta grid is chosen for stability across the $-50$ to $+50$ MHz domain and for the $-216$ MHz anharmonicity. The 40 us square pulse scales the shaped-pulse step count by its duration ratio.

In [4]:
result = simulate_qutrit_map(
    duration_us=SQUARE_DURATION_US,
    detuning_mhz=MODEL_DETUNING_MHZ,
    rabi_mhz=RABI_MHZ,
    t1_us=T1_US,
    t_phi_us=T_PHI_US,
    anharmonicity_mhz=ANHARMONICITY_MHZ,
    num_steps_per_half=SQUARE_STEPS_PER_HALF,
    cutoff=None,
    echo=False,
)

populations = {
    r'$P_g$': result.ground,
    r'$P_e$': result.excited,
    r'$P_f$': result.second_excited,
}
np.testing.assert_allclose(sum(populations.values()), 1.0, atol=2e-6)
print('Maximum populations:', {name: float(values.max()) for name, values in populations.items()})

Maximum populations: {'$P_g$': 1.0, '$P_e$': 0.5148389615411576, '$P_f$': 0.1485631302329149}


In [5]:
protocol_results = {'Constant': result}
for protocol, echo in [('Root-Lorentzian', False), ('Echo-root-Lorentzian', True)]:
    protocol_results[protocol] = simulate_qutrit_map(
        duration_us=SHAPED_DURATION_US,
        detuning_mhz=MODEL_DETUNING_MHZ,
        rabi_mhz=RABI_MHZ,
        t1_us=T1_US,
        t_phi_us=T_PHI_US,
        anharmonicity_mhz=ANHARMONICITY_MHZ,
        num_steps_per_half=SHAPED_STEPS_PER_HALF,
        cutoff=SHAPED_CUTOFF,
        echo=echo,
        order=SHAPED_ORDER,
    )
    check = protocol_results[protocol]
    np.testing.assert_allclose(check.ground + check.excited + check.second_excited, 1.0, atol=2e-6)
    print(protocol, 'maximum excitation:', float((check.excited + check.second_excited).max()))

Root-Lorentzian maximum excitation: 0.8657722625883464
Echo-root-Lorentzian maximum excitation: 0.9019775887573495


## Dressed resonance center

A finite 40 us square trace contains coherent Rabi fringes, so its largest sampled population is not a stable definition of resonance. We therefore locate the minimum avoided-crossing gap of the same three-state rotating-frame Hamiltonian. This gives a smooth resonance center; the time-domain populations below show the actual finite-pulse response.

In [6]:
def dressed_resonance_center_mhz(rabi_mhz, anharmonicity_mhz):
    # Fine local grid around the g-e avoided crossing. For negative alpha,
    # the f-like dressed branch is the lowest eigenvalue and the g/e gap
    # is the separation between the two upper eigenvalues.
    local_detuning = np.linspace(-30.0, 15.0, 9001)
    centers = np.zeros_like(rabi_mhz, dtype=float)
    gaps = np.zeros_like(rabi_mhz, dtype=float)
    for index, omega in enumerate(rabi_mhz):
        hamiltonian = np.zeros((local_detuning.size, 3, 3), dtype=float)
        hamiltonian[:, 1, 1] = -local_detuning
        hamiltonian[:, 2, 2] = -2.0 * local_detuning + anharmonicity_mhz
        hamiltonian[:, 0, 1] = hamiltonian[:, 1, 0] = omega / 2.0
        hamiltonian[:, 1, 2] = hamiltonian[:, 2, 1] = omega / np.sqrt(2.0)
        eigenvalues = np.linalg.eigvalsh(hamiltonian)
        ge_gap = eigenvalues[:, 2] - eigenvalues[:, 1]
        minimum = int(np.argmin(ge_gap))
        centers[index] = local_detuning[minimum]
        gaps[index] = ge_gap[minimum]
    return centers, gaps

dressed_center_mhz, dressed_gap_mhz = dressed_resonance_center_mhz(
    RABI_MHZ, ANHARMONICITY_MHZ
)
perturbative_center_mhz = -RABI_MHZ**2 / (2.0 * ANHARMONICITY_MHZ)

print(f'At 40 MHz peak Rabi: center = {dressed_center_mhz[np.argmin(abs(RABI_MHZ - 40))]:.3f} MHz')
print(f'At 60 MHz peak Rabi: center = {dressed_center_mhz[-1]:.3f} MHz')

At 40 MHz peak Rabi: center = 3.450 MHz
At 60 MHz peak Rabi: center = 7.190 MHz


In [7]:
# A fine central scan resolves the operational f01 feature for the shaped pulses.
CENTER_DETUNING_MHZ = np.linspace(-10.0, 10.0, 401)
central_results = {}
for protocol, duration_us, steps_per_half, cutoff, echo in [
    ('Constant', SQUARE_DURATION_US, SQUARE_STEPS_PER_HALF, None, False),
    ('Root-Lorentzian', SHAPED_DURATION_US, SHAPED_STEPS_PER_HALF, SHAPED_CUTOFF, False),
    ('Echo-root-Lorentzian', SHAPED_DURATION_US, SHAPED_STEPS_PER_HALF, SHAPED_CUTOFF, True),
]:
    central_results[protocol] = simulate_qutrit_map(
        duration_us=duration_us, detuning_mhz=CENTER_DETUNING_MHZ, rabi_mhz=RABI_MHZ,
        t1_us=T1_US, t_phi_us=T_PHI_US, anharmonicity_mhz=ANHARMONICITY_MHZ,
        num_steps_per_half=steps_per_half, cutoff=cutoff, echo=echo, order=SHAPED_ORDER,
    )

def central_feature_centers(values, *, minimum):
    centers = np.zeros(len(RABI_MHZ))
    for index, row in enumerate(values):
        if index == 0:
            continue
        spline = CubicSpline(CENTER_DETUNING_MHZ, row)
        objective = (lambda x: float(spline(x))) if minimum else (lambda x: -float(spline(x)))
        centers[index] = minimize_scalar(
            objective, bounds=(-0.5, 0.5), method='bounded', options={'xatol': 1e-9}
        ).x
    return centers

central_excitation = {
    name: values.excited + values.second_excited for name, values in central_results.items()
}
root_feature_center_mhz = central_feature_centers(central_excitation['Root-Lorentzian'], minimum=False)
echo_feature_center_mhz = central_feature_centers(central_excitation['Echo-root-Lorentzian'], minimum=True)

# Weak-drive phase-average estimate. The echo sign drops out because the Stark shift is quadratic.
sigma_us = (SHAPED_DURATION_US / 2.0) / np.sqrt(SHAPED_CUTOFF ** (-1.0 / SHAPED_ORDER) - 1.0)
mean_square_envelope = (2.0 * sigma_us / SHAPED_DURATION_US) * np.arctan(SHAPED_DURATION_US / (2.0 * sigma_us))
shaped_phase_average_mhz = mean_square_envelope * perturbative_center_mhz
for omega in (20.0, 40.0, 60.0):
    index = int(np.argmin(abs(RABI_MHZ - omega)))
    print(
        f'{omega:4.0f} MHz: constant dressed {dressed_center_mhz[index]:+.4f} MHz; '
        f'root feature {root_feature_center_mhz[index]:+.4f} MHz; '
        f'echo feature {echo_feature_center_mhz[index]:+.4f} MHz; '
        f'shaped phase-average {shaped_phase_average_mhz[index]:+.4f} MHz'
    )

  20 MHz: constant dressed +0.9100 MHz; root feature +0.0017 MHz; echo feature +0.0027 MHz; shaped phase-average +0.0029 MHz
  40 MHz: constant dressed +3.4500 MHz; root feature -0.0011 MHz; echo feature -0.0014 MHz; shaped phase-average +0.0116 MHz
  60 MHz: constant dressed +7.1900 MHz; root feature +0.0778 MHz; echo feature -0.0115 MHz; shaped phase-average +0.0261 MHz


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharex=True, sharey=True, constrained_layout=True)
images = []
for ax, (label, values) in zip(axes, populations.items()):
    image = ax.pcolormesh(
        MODEL_DETUNING_MHZ, RABI_MHZ, values, shading='auto',
        cmap='magma', vmin=0.0, vmax=1.0, rasterized=True,
    )
    images.append(image)
    ax.plot(dressed_center_mhz, RABI_MHZ, color='cyan', lw=1.4, ls='--', label='dressed center')
    ax.axvline(0.0, color='white', lw=0.7, alpha=0.65)
    ax.set_title(label)
    ax.set_xlabel(r'Drive detuning $\Delta/2\pi=f_d-f_{01}$ (MHz)')
    ax.set_xlim(MODEL_DETUNING_MHZ[0], MODEL_DETUNING_MHZ[-1])
axes[0].set_ylabel(r'Peak Rabi frequency $\Omega_0/2\pi$ (MHz)')
axes[-1].legend(loc='upper left', fontsize=7)
colorbar = fig.colorbar(images[-1], ax=axes, pad=0.02, fraction=0.035)
colorbar.set_label('Final population')
fig.suptitle('40 us square pulse: all three transmon-state populations over 100 MHz')
plt.show()

/var/folders/5l/mx_yndbx4sq978lx9ht2346r0000gn/T/ipykernel_68016/2219297333.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5), constrained_layout=True)

# Zoom into the central g-e feature so the displacement is visible.
excitation = result.excited + result.second_excited
zoom = axes[0].pcolormesh(
    MODEL_DETUNING_MHZ, RABI_MHZ, excitation, shading='auto',
    cmap='viridis', vmin=0.0, vmax=float(excitation.max()), rasterized=True,
)
axes[0].plot(dressed_center_mhz, RABI_MHZ, color='white', lw=2.0, label='exact dressed center')
axes[0].plot(perturbative_center_mhz, RABI_MHZ, color='tab:red', lw=1.2, ls='--', label=r'$-\Omega^2/(2\alpha)$')
axes[0].axvline(0.0, color='white', lw=0.7, alpha=0.65)
axes[0].set_xlim(-8.0, 12.0)
axes[0].set_xlabel(r'Drive detuning $\Delta/2\pi=f_d-f_{01}$ (MHz)')
axes[0].set_ylabel(r'$\Omega_0/2\pi$ (MHz)')
axes[0].set_title(r'Central excitation $P_e+P_f$')
axes[0].legend(loc='upper right', fontsize=7)
fig.colorbar(zoom, ax=axes[0], pad=0.02, label='Final excitation')

axes[1].plot(RABI_MHZ, dressed_center_mhz, 'o-', ms=3.0, label='exact three-state dressed center')
axes[1].plot(RABI_MHZ, perturbative_center_mhz, '--', label=r'weak-drive $-\Omega^2/(2\alpha)$')
axes[1].axhline(0.0, color='0.5', lw=0.8)
axes[1].fill_between(RABI_MHZ, 0.0, dressed_center_mhz, color='tab:blue', alpha=0.12)
axes[1].set_xlabel(r'$\Omega_0/2\pi$ (MHz)')
axes[1].set_ylabel(r'Resonance center $\Delta_{\rm res}/2\pi$ (MHz)')
axes[1].set_title('The dressed g-e resonance moves right')
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=7)
plt.show()

/var/folders/5l/mx_yndbx4sq978lx9ht2346r0000gn/T/ipykernel_68016/946934331.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
full_excitation = {
    name: values.excited + values.second_excited for name, values in protocol_results.items()
}
stable_shaped = RABI_MHZ >= 20.0
root_overlay = np.where(stable_shaped, root_feature_center_mhz, np.nan)
echo_overlay = np.where(stable_shaped, echo_feature_center_mhz, np.nan)
overlays = [dressed_center_mhz, root_overlay, echo_overlay]

fig_domains, axes = plt.subplots(2, 3, figsize=(7.2, 5.0), sharey=True, constrained_layout=True)
for column, (protocol, center) in enumerate(zip(full_excitation, overlays)):
    full_image = axes[0, column].pcolormesh(
        MODEL_DETUNING_MHZ, RABI_MHZ, full_excitation[protocol], shading='auto',
        cmap='viridis', vmin=0.0, vmax=1.0, rasterized=True,
    )
    axes[0, column].axvline(0.0, color='white', lw=0.6, ls=':', alpha=0.8)
    axes[0, column].set_xlim(-50.0, 50.0)
    axes[0, column].set_title(f'({chr(97 + column)}) {protocol}: 100 MHz', fontsize=8)
    axes[0, column].set_xlabel(r'$\Delta/2\pi$ (MHz)')

    axes[1, column].pcolormesh(
        CENTER_DETUNING_MHZ, RABI_MHZ, central_excitation[protocol], shading='auto',
        cmap='viridis', vmin=0.0, vmax=1.0, rasterized=True,
    )
    axes[1, column].plot(center, RABI_MHZ, color='white', lw=1.2)
    axes[1, column].axvline(0.0, color='white', lw=0.6, ls=':', alpha=0.8)
    axes[1, column].set_xlim(-10.0, 10.0)
    axes[1, column].set_title(f'({chr(100 + column)}) {protocol}: 20 MHz zoom', fontsize=8)
    axes[1, column].set_xlabel(r'$\Delta/2\pi$ (MHz)')
axes[0, 0].set_ylabel(r'$\Omega_0/2\pi$ (MHz)')
axes[1, 0].set_ylabel(r'$\Omega_0/2\pi$ (MHz)')
fig_domains.colorbar(full_image, ax=axes, pad=0.01, label=r'$P_e+P_f$', shrink=0.92)
saved_domains = save_figure(
    fig_domains, '11_ac_stark_qutrit_10us', variant=FigureVariant.PAPER,
    formats=('pdf', 'png', 'svg'), dpi=300, bbox_inches='tight', pad_inches=0.04,
)
print('Saved:', *(str(path.relative_to(PROJECT_ROOT)) for path in saved_domains), sep='\n  ')
plt.show()

fig_shift = plt.figure(figsize=(7.2, 5.1), constrained_layout=True)
shift_grid = fig_shift.add_gridspec(2, 2, height_ratios=(1.0, 0.9))
one_mhz_axes = [fig_shift.add_subplot(shift_grid[0, column]) for column in range(2)]
shift_axis = fig_shift.add_subplot(shift_grid[1, :])
one_mhz = abs(CENTER_DETUNING_MHZ) <= 0.5
for panel, (axis, protocol, center) in enumerate([
    (one_mhz_axes[0], 'Root-Lorentzian', root_overlay),
    (one_mhz_axes[1], 'Echo-root-Lorentzian', echo_overlay),
]):
    axis.pcolormesh(
        CENTER_DETUNING_MHZ[one_mhz], RABI_MHZ, central_excitation[protocol][:, one_mhz],
        shading='auto', cmap='viridis', vmin=0.0, vmax=1.0, rasterized=True,
    )
    axis.plot(center, RABI_MHZ, color='white', lw=1.2)
    axis.axvline(0.0, color='white', lw=0.6, ls=':', alpha=0.8)
    axis.set_xlim(-0.5, 0.5)
    axis.set_title(f'({chr(97 + panel)}) {protocol}: 1 MHz zoom', fontsize=8)
    axis.set_xlabel(r'$\Delta/2\pi$ (MHz)')
one_mhz_axes[0].set_ylabel(r'$\Omega_0/2\pi$ (MHz)')

shift_axis.plot(RABI_MHZ, dressed_center_mhz, 'o-', ms=2.5, label='constant: exact dressed $f_{01}$')
shift_axis.plot(RABI_MHZ[stable_shaped], root_feature_center_mhz[stable_shaped], 's-', ms=2.8, label='root: spectral maximum')
shift_axis.plot(RABI_MHZ[stable_shaped], echo_feature_center_mhz[stable_shaped], '^-', ms=2.8, label='echo-root: central minimum')
shift_axis.plot(RABI_MHZ[stable_shaped], shaped_phase_average_mhz[stable_shaped], 'k--', lw=1.0, label='shaped phase average')
shift_axis.axhline(0.0, color='0.5', lw=0.7)
shift_axis.set_xlabel(r'$\Omega_0/2\pi$ (MHz)')
shift_axis.set_ylabel(r'$f_{01}$ shift (MHz)')
shift_axis.set_title(r'(c) $f_{01}$ shift for all three protocols', fontsize=8)
shift_axis.grid(alpha=0.25)
shift_axis.legend(fontsize=6.2, ncol=2, loc='upper left')
inset = shift_axis.inset_axes([0.55, 0.23, 0.42, 0.55])
inset.plot(RABI_MHZ[stable_shaped], root_feature_center_mhz[stable_shaped], 's-', ms=2.0)
inset.plot(RABI_MHZ[stable_shaped], echo_feature_center_mhz[stable_shaped], '^-', ms=2.0)
inset.plot(RABI_MHZ[stable_shaped], shaped_phase_average_mhz[stable_shaped], 'k--', lw=0.8)
inset.axhline(0.0, color='0.5', lw=0.6)
inset.set_xlim(20.0, 60.0)
inset.set_ylim(-0.25, 0.25)
inset.set_title('shaped-pulse zoom', fontsize=6)
inset.tick_params(labelsize=5)
inset.grid(alpha=0.2)
saved_shift = save_figure(
    fig_shift, '12_ac_stark_f01_zoom_shift', variant=FigureVariant.PAPER,
    formats=('pdf', 'png', 'svg'), dpi=300, bbox_inches='tight', pad_inches=0.04,
)
print('Saved:', *(str(path.relative_to(PROJECT_ROOT)) for path in saved_shift), sep='\n  ')
plt.show()

Saved:
  figures/paper/11_ac_stark_qutrit_10us.pdf
  figures/paper/11_ac_stark_qutrit_10us.png
  figures/paper/11_ac_stark_qutrit_10us.svg


/var/folders/5l/mx_yndbx4sq978lx9ht2346r0000gn/T/ipykernel_68016/630877852.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved:
  figures/paper/12_ac_stark_f01_zoom_shift.pdf
  figures/paper/12_ac_stark_f01_zoom_shift.png
  figures/paper/12_ac_stark_f01_zoom_shift.svg


/var/folders/5l/mx_yndbx4sq978lx9ht2346r0000gn/T/ipykernel_68016/630877852.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation

The plotted quantity for the protocol comparison is the central $f_{01}$ excitation response $P_e+P_f$; $|f\rangle$ is retained only as the virtual level that produces the multilevel shift. With the conventional $\Delta=f_d-f_{01}$ axis, the constant-pulse dressed $f_{01}$ avoided crossing moves to **positive** detuning (right). Its weak-drive result is approximately

$$\frac{\Delta_{\mathrm{res}}}{2\pi} \simeq -\frac{(\Omega_0/2\pi)^2}{2(\alpha/2\pi)},$$

which is positive because $\alpha<0$. At $60$ MHz peak Rabi the exact constant-pulse displacement is $+7.190$ MHz. For the root-Lorentzian and echo-root-Lorentzian pulses ($c=0.002$), the peak exists only briefly and the phase-averaged weak-drive shift is $+0.026$ MHz. The corresponding finite-pulse features at $60$ MHz are $+0.078$ MHz for the root-Lorentzian maximum and $-0.012$ MHz for the echo-root-Lorentzian central minimum. These operational feature positions include coherent finite-pulse dynamics and therefore need not equal the phase-average estimate or share its sign.